In [64]:
import asyncio # Imports the library for running asynchronous code.
from agents import Agent, Runner, set_tracing_disabled # Imports the core components from the OpenAI Agents SDK.
import os 
# Disable the SDK's tracing feature to keep the output clean for this tutorial.
set_tracing_disabled(True)

LITELLM_API_BASE = os.environ.get("LITELLM_API_BASE")
NEBIUS_API_KEY = os.environ.get("NEBIUS_API_KEY")
# Define a simple agent for a quick test.
agent = Agent(
    name="Assistant", # Assign a name to the agent.
    instructions="Reply very concisely.", # Provide a simple instruction to guide its behavior.
    # Specify the model using the LiteLLM provider format for Nebius.
    model="litellm/nebius/moonshotai/Kimi-K2-Instruct",
)

# Execute the agent with a test prompt. 'await' is used because the run is an asynchronous operation.
result = await Runner.run(
    agent, # The agent to run.
    "Tell me why it is important to evaluate AI agents." # The user's input prompt.
)

# Print the final text output from the agent's response.
print(result.final_output)

Evaluation reveals hidden biases, reliability gaps, and safety risks while guiding iterative improvements, preventing downstream harm, and fostering trust among users and regulators.


In [65]:
from openai import OpenAI 

client = OpenAI(
    # Sets the base URL for the API endpoint to the Nebius service.
    base_url=os.environ.get("LITELLM_API_BASE_URL"),
    # Sets the API key for authentication. Replace with your actual key, preferably loaded from a secure source.
    api_key=os.environ.get("OPENAI_API_KEY")
)

In [66]:
from dataclasses import dataclass, field
from typing import Any, Dict, List

@dataclass
class MemoryNote:
    text: str
    last_update_date: str
    keywords: List[str]


In [67]:
@dataclass
class TravelState:
    profile: Dict[str, Any] = field(default_factory=dict)

    global_memory: Dict[str, Any] = field(default_factory=lambda: {"notes": []})
    session_memory: Dict[str, Any] = field(default_factory=lambda: {"notes": []})
    trip_history: Dict[str, Any] = field(default_factory=lambda: {"trips":[]})

    system_frontmatter: str = ""
    global_memories_md: str = ""
    session_memories_md:str = ""

    inject_session_memories_next_turn: bool = False

In [68]:
user_state = TravelState(
    profile={
        "global_customer_id": "crm_12345",
        "name": "John Doe",
        "age": "31",
        "home_city": "San Francisco",
        "currency" : "USD",
        "passport_expiry_date": "2029-06-12",
        "loyalty_status": {"airline": "United Gold", "hotel": "Marriott Titanium"},
        "loyalty_ids": {"marriott": "MR998877", "hilton": "HH445566", "hyatt": "HY112233"},
        "seat_preference": "aisle",
        "tone": "concise and friendly",
        "active_visas": ["Schengen", "US"],
        "insurance_coverage_profile": {
            "car_rental": "primary_cdw_included",
            "travel_medical": "covered",
        },
    },
    global_memory = {
        "notes": [
            # Each note is an instance of MemoryNote, converted to a dictionary for JSON compatibility.
            MemoryNote(
                text="For trips shorter than a week, user generally prefers not to check bags.",
                last_update_date="2025-04-05",
                keywords=["baggage", "short_trip"],
            ).__dict__,
            MemoryNote(
                text="User usually prefers aisle seats.",
                last_update_date="2024-06-25",
                keywords=["seat_preference"],
            ).__dict__,
            MemoryNote(
                text="User generally likes central, walkable city-center neighborhoods.",
                last_update_date="2024-02-11",
                keywords=["neighborhood"],
            ).__dict__,
            MemoryNote(
                text="User generally likes to compare options side-by-side",
                last_update_date="2023-02-17",
                keywords=["pricing"],
            ).__dict__,
            MemoryNote(
                text="User prefers high floors",
                last_update_date="2023-02-11",
                keywords=["room"],
            ).__dict__,
        ]
    },
    trip_history = {
        "trips": [
            {
                # Core trip details
                "from_city": "Istanbul",
                "from_country": "Turkey",
                "to_city": "Paris",
                "to_country": "France",
                "check_in_date": "2025-05-01",
                "check_out_date": "2025-05-03",
                "trip_purpose": "leisure",  # leisure | business | family | etc.
                "party_size": 1,

                # Flight details
                "flight": {
                    "airline": "United",
                    "airline_status_at_booking": "United Gold",
                    "cabin_class": "economy_plus",
                    "seat_selected": "aisle",
                    "seat_location": "front",          # front | middle | back
                    "layovers": 1,
                    "baggage": {"checked_bags": 0, "carry_ons": 1},
                    "special_requests": ["vegetarian_meal"],  # optional
                },

                # Hotel details
                "hotel": {
                    "brand": "Hilton",
                    "property_name": "Hilton Paris Opera",
                    "neighborhood": "city_center",
                    "bed_type": "king",
                    "smoking": "non_smoking",
                    "high_floor": True,
                    "early_check_in": False,
                    "late_check_out": True,
                },
            }
        ]
    },
)

/opt/anaconda3/lib/python3.13/collections/__init__.py:450: RuntimeWarning: coroutine 'TrimmingSession._trim_to_last_turns' was never awaited
  @classmethod


In [69]:
from datetime import datetime,timezone
from typing import List
from agents import function_tool,RunContextWrapper


def _today_iso_utc() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT")


@function_tool
def save_memory_note(ctx: RunContextWrapper[TravelState],text:str, keywords : List[str]) -> dict:
    if "notes" not in ctx.context.session_memory or ctx.context.session_memory["notes"] is None:
        ctx.context.session_memory["notes"] = []

    clean_keywords = [
        k.strip().lower()
        for k in keywords if isinstance(k,str) and k.strip()
        ][:3]

    ctx.context.session_memory["notes"].append({
        "text": text.strip(),
        "last_update_date": _today_iso_utc(),
        "keywords": clean_keywords
    })
    return {"ok":True}


In [70]:
from __future__ import annotations
import asyncio
from collections import deque
from typing import Any, Deque, List, cast
from agents.memory.session import SessionABC
from agents.items import TResponse, TResponseInputItem

ROLE_USER = "user"

def _is_user_msg(item: TResponseInputItem) -> bool:
    if isinstance(item,dict):
        role = item.get('role')
        if role is not None:
            return role == ROLE_USER

        if item.get("type") == "message":
            return item.get("role") == ROLE_USER

    return getattr(item, "role", None) == ROLE_USER

class TrimmingSession(SessionABC):

    def __init__(self, session_id: str, state: TravelState, max_turns: int = 8):
        self.session_id = session_id
        self.state = state
        self.max_turns = max(1, int(max_turns))
        self._items : Deque[TResponseInputItem] = deque()
        self._lock = asyncio.Lock()

    async def get_items(self, limit:int | None = None) -> List[TResponseInputItem]:
        async with self._lock:
            trimmed = await self._trim_to_last_turns(list(self._items))
            return trimmed[-limit:] if (limit is not None and limit >=0) else trimmed

    async def add_items(self, items: List[TResponseInputItem]) -> None:
        """"""
        if not items:
            return

        async with self._lock:
            self._items.extend(items)
            original_len = len(self._items)
            trimmed = await self._trim_to_last_turns(list(self._items))

            if len(trimmed) < original_len:
                self.state.inject_session_memories_next_turn = True

            self._items.clear()
            self._items.extend(trimmed)

    async def pop_item(self):
        async with self._lock:
            return self._items.pop()

    async def clear_session(self) -> None:
        async with self._lock:
            self._items.clear()

    async def _trim_to_last_turns(self,items: List[TResponseInputItem]) -> List[TResponseInputItem]:
        if not items:
            return items

        count = 0
        start_idx = 0

        for i in range(len(items) - 1, -1, -1):
            if _is_user_msg(items[i]):
                count +=1
                if count == self.max_turns:
                    start_idx = i
                    break 
        return items[start_idx:]
        





In [71]:
session = TrimmingSession("my_session", user_state, max_turns=20)

In [72]:
# Define a multi-line string containing the rules and guidelines for how the agent should use memory.
MEMORY_INSTRUCTIONS = """
<memory_policy>
You may receive two memory lists:
- GLOBAL memory = long-term defaults (“usually / in general”).
- SESSION memory = trip-specific overrides (“this trip / this time”).

How to use memory:
- Use memory only when it is relevant to the user’s current decision (flight/hotel/insurance choices).
- Apply relevant memory automatically when setting tone, proposing options and making recommendations.
- Do not repeat memory verbatim to the user unless it’s necessary to confirm a critical constraint.

Precedence and conflicts:
1) The user’s latest message in this conversation overrides everything.
2) SESSION memory overrides GLOBAL memory for this trip when they conflict.
   - Example: GLOBAL “usually aisle” + SESSION “this time window to sleep” ⇒ choose window for this trip.
3) Within the same memory list, if two items conflict, prefer the most recent by date.
4) Treat GLOBAL memory as a default, not a hard constraint, unless the user explicitly states it as non-negotiable.

When to ask a clarifying question:
- Ask exactly one focused question only if a memory materially affects booking and the user’s intent is ambiguous.
  (e.g., “Do you want to keep the window seat preference for all legs or just the overnight flight?”)

Where memory should influence decisions (check these before suggesting options):
- Flights: seat preference, baggage habits (carry-on vs checked), airline loyalty/status, layover tolerance if mentioned.
- Hotels: neighborhood/location style (central/walkable), room preferences (high floor), brand loyalty IDs/status.
- Insurance: known coverage profile (e.g., CDW included) and whether the user wants add-ons this trip.

Memory updates:
- Do NOT treat “this time” requests as changes to GLOBAL defaults.
- Only promote a preference into GLOBAL memory if the user indicates it’s a lasting rule
  (e.g., “from now on”, “generally”, “I usually prefer X now”).
- If a new durable preference/constraint appears, store it via the memory tool (short, general, non-PII).

Safety:
- Never store or echo sensitive PII (passport numbers, payment details, full DOB).
- If a memory seems stale or conflicts with user intent, defer to the user and proceed accordingly.
</memory_policy>
"""

In [73]:
# Define a multi-line string containing the rules and guidelines for how the agent should use memory.
MEMORY_INSTRUCTIONS = """
<memory_policy>
You may receive two memory lists:
- GLOBAL memory = long-term defaults (“usually / in general”).
- SESSION memory = trip-specific overrides (“this trip / this time”).

How to use memory:
- Use memory only when it is relevant to the user’s current decision (flight/hotel/insurance choices).
- Apply relevant memory automatically when setting tone, proposing options and making recommendations.
- Do not repeat memory verbatim to the user unless it’s necessary to confirm a critical constraint.

Precedence and conflicts:
1) The user’s latest message in this conversation overrides everything.
2) SESSION memory overrides GLOBAL memory for this trip when they conflict.
   - Example: GLOBAL “usually aisle” + SESSION “this time window to sleep” ⇒ choose window for this trip.
3) Within the same memory list, if two items conflict, prefer the most recent by date.
4) Treat GLOBAL memory as a default, not a hard constraint, unless the user explicitly states it as non-negotiable.

When to ask a clarifying question:
- Ask exactly one focused question only if a memory materially affects booking and the user’s intent is ambiguous.
  (e.g., “Do you want to keep the window seat preference for all legs or just the overnight flight?”)

Where memory should influence decisions (check these before suggesting options):
- Flights: seat preference, baggage habits (carry-on vs checked), airline loyalty/status, layover tolerance if mentioned.
- Hotels: neighborhood/location style (central/walkable), room preferences (high floor), brand loyalty IDs/status.
- Insurance: known coverage profile (e.g., CDW included) and whether the user wants add-ons this trip.

Memory updates:
- Do NOT treat “this time” requests as changes to GLOBAL defaults.
- Only promote a preference into GLOBAL memory if the user indicates it’s a lasting rule
  (e.g., “from now on”, “generally”, “I usually prefer X now”).
- If a new durable preference/constraint appears, store it via the memory tool (short, general, non-PII).

Safety:
- Never store or echo sensitive PII (passport numbers, payment details, full DOB).
- If a memory seems stale or conflicts with user intent, defer to the user and proceed accordingly.
</memory_policy>
"""

In [74]:
from pickle import TRUE
import yaml


def render_frontmatter(profile: dict) -> str:
    payload = {"profile":profile}
    y = yaml.safe_dump(payload, sort_keys = False).strip()
    return f"---\n{y}\n---"


def render_global_memories_md(global_notes: list[dict], k:int = 6) -> str:
    if not global_notes:
        return "- (none)"

    notes_sorted = sorted(global_notes, key=lambda n:n.get("last_update_date",""), reverse=True)
    top = notes_sorted[:k]
    return "\n".join([f"- {n["text"]}" for n in top])


def render_session_memories_md(session_notes: list[dict], k:int = 8) -> str:
    if not session_notes:
        return "- (none)"

    top = session_notes[-k:]
    return "\n".join([f"- {n['text']}" for n in top])

In [75]:
from agents import AgentHooks, Agent


class MemoryHooks(AgentHooks[TravelState]):

    def __init__(self, client: client):
        self.client = client

    async def on_start(self, ctx: RunContextWrapper[TravelState], agent:Agent) -> None:
        ctx.context.system_frontmatter = render_frontmatter(ctx.context.profile)
        ctx.context.global_memories_md = render_global_memories_md((ctx.context.global_memory or {}).get("notes", []))


        if ctx.context.inject_session_memories_next_turn:
            ctx.context.session_memories_md = render_session_memories_md(
                (ctx.context.session_memory or {}).get("notes",[])
            )
        else:
            ctx.context.session_memories_md = ""


In [76]:
# Define the base system prompt for the travel concierge agent.
BASE_INSTRUCTIONS = f"""
You are a concise, reliable travel concierge. 
Help users plan and book flights, hotels, and car/travel insurance.\n\n
Guidelines:\n
- Collect key trip details and confirm understanding.\n
- Ask only one focused clarifying question at a time.\n
- Provide a few strong options with brief tradeoffs, then recommend one.\n
- Respect stable user preferences and constraints; avoid assumptions.\n
- Before booking, restate all details and get explicit approval.\n
- Never invent prices, availability, or policies—use tools or state uncertainty.\n
- Do not repeat sensitive PII; only request what is required.\n
- Track multi-step itineraries and unresolved decisions.\n\n
"""

In [77]:
async def instructions(ctx: RunContextWrapper[TravelState], agent:Agent) -> str:
    s = ctx.context

    if s.inject_session_memories_next_turn and not s.session_memories_md:
        s.session_memories_md = render_session_memories_md(
            (s.session_memory or {}).get("notes",[])
        )

    session_block = ""

    if s.inject_session_memories_next_turn and s.session_memories_md:
        session_block = (
            "\n\nSESSION memory (temporary: overrides GLOBAL when conflicting):\n"
            + s.session_memories_md
        )

        s.inject_session_memories_next_turn = False
        s.session_memories_md = ""

    return (
        BASE_INSTRUCTIONS
        +"\n\n<user_profile\n"+ (s.system_frontmatter or "") + "\n</user_profile>"
        +"\n\n<memories>\n"
        +"GLOBAL memory: \n" + (s.global_memories_md or "- (none)")
        +session_block
        +"\n</memories>"
        +"\n\n" + MEMORY_INSTRUCTIONS
    )

In [78]:
travel_concierage_agent = Agent(
    name= "Travel Concierage",
    model = "litellm/nebius/moonshotai/Kimi-K2-Instruct",
    instructions = instructions,
    hooks = MemoryHooks(client),
    tools = [save_memory_note]
)

In [79]:
import inspect

print(type(user_state), inspect.iscoroutine(user_state))
print(type(session), inspect.iscoroutine(session))

<class '__main__.TravelState'> False
<class '__main__.TrimmingSession'> False


In [80]:
r1 = await Runner.run(
    travel_concierage_agent,
    input = "Book me a flight to Paris next month",
    session = session,
    context = user_state
)

In [81]:
# Execute the second turn of the conversation.
r2 = await Runner.run(
    travel_concierage_agent, # The agent to run.
    input="Do you know my preferences?", # The user's input, asking about its knowledge.
    session=session, # The same session object, which now contains Turn 1.
    context=user_state, # The same state object.
)
# Print the agent's final text output for this turn.
print("\nTurn 2:", r2.final_output)


Turn 2: Yes - I have your preferences on file:
- **Aisle seats**  
- **Carry-on only** for short trips (under a week)  
- **United Gold status** (I'll prioritize United and Star Alliance)  
- **Central, walkable neighborhoods** for hotels (for when we get to accommodation)

But I still need your **specific travel dates** for the Paris trip. What departure and return dates work best for you next month?


In [83]:
def consolidate_memory(state: TravelState, client, model:str = "moonshotai/Kimi-K2-Instruct"):
    f"""
    You are consolidating travel memory notes in LONG-TERM(GLOBAL) memory

    You will receive two JSON arrays:
    -GLOBAL_NOTES: existing long-term notes
    -SESSION_NOTES: new notes captured during this run


    GOAL
    Produce an updated GLOBAL_NOTES list by merging in SESSION_NOTES

    RULES
    1) Keep only durable information (preferences, stable constraints, memberships/IDs, long-lived habits)
    2) Drop session-only / ephermal notes. In particular, DO NOT ADD any note if its clearly for this specific trip
    3) De-duplicates:
    -Remove the exact duplicates
    - Remove near-duplicates having same mearning. Keep the single best canonical version
    4) Conflict resolution:
    - If two notes conflict, keep the one with most recent last_update_date(YYYY-MM-DD)
    - If dates tie, prefer SESSION_NOTES over GLOBAL_NOTES
    5) Note quality:
    - Keep each note short (1 sentence), specific and durable
    - Prefer canonical phrasing like: "Prefers aisle seats." / "Avoids red-eye flights." / "Has United Gold status"
    6) DO NOT invent new facts. Only use what appears in the input notes

    OUTPUT FORMAT (STRICT)
    Return ONLY a valid JSON array.
    Each element MUST be an object with EXACTLY these keys:
    {{"text": string, "last_update_date":"YYYY-MM-DD", "keywords": [string]}}

    Do not include markdown, commentary, code fences, or extra keys.

    GLOBAL_NOTES (JSON):
    <GLOBAL_JSON>
    {global_json}
    </GLOBAL_JSON>

    SESSION_NOTES (JSON)
    <SESSION_JSON>
    {session_json}
    </SESSION_JSON>
    """.strip()

    import json

    session_notes: List[Dict[str, Any]] = state.session_memory.get("notes",[]) or []
    if not session_notes:
        return


    global_notes: List[Dict[str,Any]] = state.global_memory.ge("notes",[]) or []
    global_json = json.dumps(global_notes, ensure_ascii = False)
    session_json = json.dumps(session_notes, ensure_ascii=False)


    consolidation_prompt = """"""

    
    resp = client.chat.completions.create(
        model= model,
        messages = [
            { "role":"user", "content":consolidation_prompt}
        ],
        temperature = 0.0
    )

    consolidated_text = resp.choices[0].message.content.strip()

    try:
        if "```json" in consolidated_text:
            consolidated_text = consolidated_text.split("```json")[1].split("```")[0].strip()
        elif "```" in consolidated_text:
            consolidated_text = consolidated_text.split("```")[1].split("```")[0].strip()
            
        consolidated_notes = json.loads(consolidated_text)

        if isinstance(consolidated_notes, list):
            state.global_memory["notes"] = consolidated_notes
            print("new global memory count", len(consolidated_notes))
        else:
            state.global_memory["notes"] = global_notes + session_notes
    except Exception as e:
        state.global_memory["notes"] = global_notes + session_notes

    state.session_memory["notes"]= []

## Guardrails and Evaluation ##

In [84]:
import re


def contains_senstive_info(text:str) -> bool:
    cc_pattern = r'\b(?:\d[ -]*?){13,16}\b'
    id_pattern = r'\b[A-Z0-9]{7,12}\b'

    if re.search(cc_pattern, text):
        return True
    return False

In [ ]:
@function_tool
def delete_memory_note(ctx: RunContextWrapper[TravelState], keyword:str) -> dict:

    """
    Remove a specific preference from long-term memory if the user no longer wants it.
    Example: 'I don't care about aisle seats anymore' -> keyword = 'seat'
    """

    initial_count = len(ctx.context.global_memory["notes"])
    ctx.content.global_memory["notes"] = [
        n for n in ctx.context.global_memory["notes"]
        if keyword.lower() not in [k.lower() for k in n.get("keywords",[])]
        and keyword.lower() not in n["text"].lower()
    ]

    removed = initial_count - len(ctx.context.global_memory["notes"])
    return {"status":"success", "notes_removed":removed}


In [ ]:
@function_tool
def save_memory_note_safe(ctx: RunContextWrapper[TravelState], text:str, keywords: List[str]) -> dict:
    """Save a durable preference. Blocks sensitive PII automatically"""

    if contains_senstive_info(text):
        return {"ok": False, "error":"Don't store sensitive financial information"}

    clean_keywords = [k.strip().lower() for k in keywords if isinstance(k, str)[:3]]
    ctx.context.session_memory["notes"].append({
        "text": text.strip(),
        "last_update_date": _today_iso_utc(),
        "keywords": clean_keywords
    })

    return {"ok":True}